In [ ]:
"""
Task 3: Cuisine Classification

Predict a restaurant's primary cuisine (the first item in the comma-separated
`Cuisines` field) from its other features.

We restrict to the top-N most common primary cuisines — the long tail has
hundreds of classes with only a handful of samples each, which hurts both
training and evaluation. That tail is reported separately as "Other" so we
can see how much signal we're leaving on the table.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

DATA_PATH = Path(r"D:\Code\Repo\Machine-Learning-Internship\Dataset .csv")
RANDOM_STATE = 42
TOP_N_CUISINES = 15  # keep the problem tractable


In [ ]:
def load_and_prep(path: Path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.dropna(subset=["Cuisines"]).copy()
    # Primary cuisine = first token
    df["Primary Cuisine"] = df["Cuisines"].str.split(",").str[0].str.strip()

    # Drop leakage / identifiers / free text
    drop_cols = [
        "Restaurant ID", "Restaurant Name", "Address",
        "Locality Verbose", "Cuisines",  # leakage: primary is derived from it
        "Rating color", "Rating text",
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Fill remaining NaNs
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].fillna("Unknown")
    return df


In [ ]:
def filter_top_cuisines(df: pd.DataFrame, n: int):
    top = df["Primary Cuisine"].value_counts().head(n).index.tolist()
    kept = df[df["Primary Cuisine"].isin(top)].copy()
    dropped = len(df) - len(kept)
    print(f"Keeping top {n} cuisines: {len(kept)} rows ({dropped} tail rows excluded)")
    print("Class distribution (train+test):")
    print(kept["Primary Cuisine"].value_counts().to_string())
    return kept, top


In [ ]:
def encode(df: pd.DataFrame):
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        if col == "Primary Cuisine":
            continue
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df


In [ ]:
def main():
    df = load_and_prep(DATA_PATH)
    df, top_cuisines = filter_top_cuisines(df, TOP_N_CUISINES)
    df = encode(df)

    y = df["Primary Cuisine"]
    X = df.drop(columns=["Primary Cuisine"])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"\nTrain: {X_train.shape}  Test: {X_test.shape}")

    # Logistic regression needs scaled features; RF does not
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    models = {
        "LogisticRegression": (
            LogisticRegression(max_iter=2000, multi_class="multinomial",
                               class_weight="balanced", n_jobs=-1,
                               random_state=RANDOM_STATE),
            X_train_s, X_test_s,
        ),
        "RandomForest": (
            RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   random_state=RANDOM_STATE, n_jobs=-1),
            X_train, X_test,
        ),
    }

    print("\n=== Overall metrics (macro-averaged) ===")
    fitted = {}
    for name, (m, Xtr, Xte) in models.items():
        m.fit(Xtr, y_train)
        pred = m.predict(Xte)
        acc = accuracy_score(y_test, pred)
        prec = precision_score(y_test, pred, average="macro", zero_division=0)
        rec = recall_score(y_test, pred, average="macro", zero_division=0)
        f1 = f1_score(y_test, pred, average="macro", zero_division=0)
        f1w = f1_score(y_test, pred, average="weighted", zero_division=0)
        print(f"  {name:20s} acc={acc:.4f}  P={prec:.4f}  R={rec:.4f}  "
              f"F1-macro={f1:.4f}  F1-weighted={f1w:.4f}")
        fitted[name] = (m, pred)

    # Detailed report for the stronger model
    best_name = max(fitted, key=lambda n: f1_score(y_test, fitted[n][1],
                                                    average="macro",
                                                    zero_division=0))
    best_model, best_pred = fitted[best_name]
    print(f"\n=== Per-cuisine report ({best_name}) ===")
    print(classification_report(y_test, best_pred, zero_division=0, digits=3))

    # Feature importances (RF only)
    if best_name == "RandomForest":
        imp = pd.Series(best_model.feature_importances_, index=X.columns
                        ).sort_values(ascending=False)
        print("Top 10 features driving cuisine predictions:")
        print(imp.head(10).to_string())

    # Confusion-matrix: show the biggest cross-cuisine confusions
    print("\n=== Top cross-cuisine confusions (true -> predicted) ===")
    labels = sorted(y_test.unique())
    cm = confusion_matrix(y_test, best_pred, labels=labels)
    confusions = []
    for i, true in enumerate(labels):
        for j, pred in enumerate(labels):
            if i != j and cm[i, j] > 0:
                confusions.append((true, pred, cm[i, j]))
    confusions.sort(key=lambda x: -x[2])
    for true, pred, n in confusions[:10]:
        print(f"  {true:25s} -> {pred:25s}  n={n}")


In [ ]:
if __name__ == "__main__":
    main()
